In [3]:
import sagemaker
from sagemaker.local import LocalSession
from sagemaker.xgboost.estimator import XGBoost

sagemaker_session = LocalSession()
sagemaker_session.config = {"local": {"local_code": True}}

role = sagemaker.get_execution_role()

estimator = XGBoost(
    entry_point="train_entry.py",
    source_dir=".",
    role=role,
    instance_type="local",       
    instance_count=1,
    framework_version="1.7-1",
    hyperparameters={
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    output_path="s3://pjm-load-model-artifacts/",
    base_job_name="pjm-load-forecast",
    sagemaker_session=sagemaker_session,
)

estimator.fit({
    "train": "s3://pjm-load-processed/train/",
    "validation": "s3://pjm-load-processed/val/",
})

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: XGBoost is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://git

 Container w45dyowuiy-algo-1-v6t2q  Creating
 Container w45dyowuiy-algo-1-v6t2q  Created
Attaching to w45dyowuiy-algo-1-v6t2q
w45dyowuiy-algo-1-v6t2q  | /miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
w45dyowuiy-algo-1-v6t2q  |   import pkg_resources
w45dyowuiy-algo-1-v6t2q  | [2026-07-21 09:56:03.925 a6545ea64aab:1 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
w45dyowuiy-algo-1-v6t2q  | [2026-07-21 09:56:04.011 a6545ea64aab:1 INFO profiler_config_parser.py:111] Unable to find config at /opt/ml/input/config/profilerconfig.json. Profiler is disabled.
w45dyowuiy-algo-1-v6t2q  | [2026-07-21:09:56:04:INFO] Imported framework sagemaker_xgboost_container.training
w45dyowuiy-algo-1-v6t2q  | [2026-07-21:09:56:04:INFO] No

INFO:sagemaker.local.image:===== Job Complete =====


In [4]:
from sagemaker.xgboost.model import XGBoostModel
from sagemaker.serverless import ServerlessInferenceConfig

MODEL_S3_URI = "s3://pjm-load-model-artifacts/pjm-load-forecast-2026-07-21-09-56-00-984/output/model.tar.gz"  

xgb_model = XGBoostModel(
    model_data=MODEL_S3_URI,
    role=role,                      
    entry_point="inference.py",
    source_dir=".",               
    framework_version="1.7-1",
)

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)

predictor = xgb_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="pjm-load-forecast-endpoint",
)

print("Endpoint deployed:", predictor.endpoint_name)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: XGBoostModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Created S3 bucket: sagemaker-us-east-1-276594885984
INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-07-21-10-17-59-578
INFO:sagemaker:Creating endpoint-config with name pjm-load-forecast-endpoint
INFO:sagemaker:Creating endpoint with name pjm-load-forecast-endpoint


------!

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: XGBoostPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint deployed: pjm-load-forecast-endpoint


In [8]:
import json
from sagemaker.deserializers import JSONDeserializer
predictor.deserializer = JSONDeserializer()

sample_request = {
    "features": [
        14,      # hour
        2,       # dayofweek (Wednesday)
        7,       # month
        3,       # quarter
        0,       # is_weekend
        0,       # is_holiday
        -0.5,    # hour_sin
        -0.866,  # hour_cos
        -0.5,    # month_sin
        -0.866,  # month_cos
        0.975,   # dow_sin
        -0.223,  # dow_cos
        30500,   # lag_24h
        31200,   # lag_168h
        29800,   # rolling_mean_168h
        1200,    # rolling_std_168h
        88.5,    # temp_f (hot summer afternoon)
        0,       # hdd
        23.5,    # cdd
    ]
}

response = predictor.predict(
    json.dumps(sample_request),
    initial_args={
        "ContentType": "application/json",
        "Accept": "application/json",  
    },
)
print(response)

{'prediction': 36854.8984375}


In [9]:
import os
os.makedirs("data/processed", exist_ok=True)

import boto3
s3 = boto3.client("s3")
s3.download_file("pjm-load-processed", "val/val.parquet", "data/processed/val.parquet")
s3.download_file("pjm-load-processed", "test/test.parquet", "data/processed/test.parquet")

print("Downloaded:", os.listdir("data/processed"))

Downloaded: ['test.parquet', 'val.parquet']


In [4]:
%run seed_dynamodb.py


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/ipykernel/__main__.py", line 6, in <module>
    app.launch_new_instance()
  File "/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/traitlets/config/application.p

AttributeError: _ARRAY_API not found

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.